In [1]:
"""
Script to run TableGAN on multiple datasets.
Complete version for all datasets: adult, car, magic, nursery, shuttle
"""

import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess


def ensure(path):
    """Create directory if it doesn't exist."""
    os.makedirs(path, exist_ok=True)


def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    """
    Create a pipeline with optional evaluation settings.
    
    Args:
        model_callable: Function that returns model instance
        skip_evaluations: If True, skip TSTR evaluations
        evaluations: List of specific evaluations to run
    """
    if skip_evaluations:
        return TrainTestSplitPipeline(
            model=model_callable, 
            evaluations=[], 
            override_evaluations=True
        )
    elif evaluations is not None:
        return TrainTestSplitPipeline(
            model=model_callable, 
            evaluations=evaluations, 
            override_evaluations=True
        )
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [2]:
# Datasets to process - ALL DATASETS
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']

# Set to False to run TSTR evaluations (requires xgboost)
# Set to True to skip evaluations for faster execution
SKIP_EVALUATIONS = False

# Create necessary directories
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

# TableGAN Configuration
TABLEGAN_CONFIG = {
    'epochs': 300,              # Number of training epochs
    'batch_size': 64,           # Batch size for training
    'noise_dim': 100,           # Dimension of noise vector
    'generator_filters': (256, 128, 64),      # Generator filter sizes
    'discriminator_filters': (64, 128, 256),  # Discriminator filter sizes
    'generator_lr': 2e-4,       # Generator learning rate
    'discriminator_lr': 2e-4,   # Discriminator learning rate
    'n_critic': 5,              # Number of discriminator updates per generator update
    'lambda_gp': 10.0,          # Gradient penalty coefficient
    'use_gradient_penalty': True,  # Use WGAN-GP
    'dropout': 0.1,             # Dropout rate
}

In [3]:
print("\n" + "=" * 80)
print("STEP 1: PREPROCESSING DATASETS")
print("=" * 80)

for dataset in DATASETS:
    print(f'\nPreprocessing {dataset}...')
    try:
        discretize_preprocess(
            file_path=f'raw_data/{dataset}.csv',
            output_path=f'discretized_data/{dataset}.csv',
            bins=10,
            strategy='uniform'
        )
        print(f'✓ Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'✗ Failed to preprocess {dataset}: {e}')
        traceback.print_exc()



STEP 1: PREPROCESSING DATASETS

Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
✓ Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
✓ Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
✓ Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
✓ Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
✓ Discretized -> discretized_data/shuttle.csv


In [ ]:

print("\n" + "=" * 80)
print("STEP 2: RUNNING TABLEGAN")
print("=" * 80)

for dataset in DATASETS:
    print(f'\n{"-" * 60}')
    print(f'Running TableGAN on {dataset}...')
    print(f'{"-" * 60}')
    
    synth_dir = os.path.join('synthetic', dataset, 'tablegan')
    ensure(synth_dir)
    
    try:
        # Import TableGAN
        module = importlib.import_module('katabatic.models.tablegan.models')
        TableGAN = getattr(module, 'TableGAN')
        
        # Create model factory
        model_factory = lambda: TableGAN(**TABLEGAN_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        
        # Run pipeline
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        
        print(f'✓ TableGAN finished for {dataset}')
        if result:
            print(f'  Result: {result}')
    except Exception as e:
        print(f'✗ TableGAN failed for {dataset}: {e}')
        traceback.print_exc()


STEP 2: RUNNING TABLEGAN

------------------------------------------------------------
Running TableGAN on adult...
------------------------------------------------------------
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64


INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Training TableGAN Model
INFO:katabatic.models.tablegan.models:================================================================================
INFO:katabatic.models.tablegan.models:Loaded training data: (26048, 14)


Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)


INFO:katabatic.models.tablegan.models:Table dimensions: 4x4 (padded from 15 features)
INFO:katabatic.models.tablegan.models:Generator parameters: 109,568
INFO:katabatic.models.tablegan.models:Discriminator parameters: 75,201
INFO:katabatic.models.tablegan.models:Epoch 1/300: D Loss = -0.852799, G Loss = -0.604852
INFO:katabatic.models.tablegan.models:Epoch 20/300: D Loss = -0.349520, G Loss = -1.264965
INFO:katabatic.models.tablegan.models:Epoch 40/300: D Loss = -0.386612, G Loss = -1.113176
INFO:katabatic.models.tablegan.models:Epoch 60/300: D Loss = -0.362978, G Loss = -0.776184
INFO:katabatic.models.tablegan.models:Epoch 80/300: D Loss = -0.310808, G Loss = -0.437272
INFO:katabatic.models.tablegan.models:Epoch 100/300: D Loss = -0.300869, G Loss = 0.041329


In [ ]:
print("\n" + "=" * 80)
print("TABLEGAN COMPLETED")
print("=" * 80)
print("\nSynthetic data has been generated for all datasets.")
print("Check the following directories:")
for dataset in DATASETS:
    print(f"  - synthetic/{dataset}/tablegan/")
print("\nResults stored in: Results/")